# Machine Learning with Spark ML – Banking Dataset

# 1Q Load the bank.csv dataset into a Spark DataFrame.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import required library
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BankMarketingAnalysis") \
    .getOrCreate()

# Read CSV file
bank_df = spark.read.csv(
    "/content/drive/MyDrive/capstone project /ml engineer/Banking_Distributed_ML_Project/bank.csv",
    header=True,
    inferSchema=True
)

# Display schema
bank_df.printSchema()

# Display sample records
bank_df.show(5)

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)

+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+

The dataset was loaded into a Spark DataFrame using SparkSession. Schema inference was enabled to automatically detect correct data types for each column.


Dataset contains 17 columns and 4521 records
Includes numerical, categorical, and target variable (y)
Data loaded successfully without null issues

## 2Q Perform basic exploratory data analysis (EDA) to understand the dataset. Display the schema,
#the first few rows, the number of rows and columns, and descriptive statistics for numeric columns.


In [3]:
# View dataset structure
bank_df.printSchema()

# Display first five rows
bank_df.show(5)

# Total rows and columns
row_total = bank_df.count()
column_total = len(bank_df.columns)

print("Total Rows:", row_total)
print("Total Columns:", column_total)

# Statistical summary of numerical columns
bank_df.describe(
    ["age", "balance", "day", "duration",
     "campaign", "pdays", "previous"]
).show()

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)

+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+

Exploratory Data Analysis was performed to understand the structure of the dataset. The dataset contains 4,521 records and 17 attributes. Summary statistics provided insights into the distribution of numerical features and confirmed that the data is suitable for machine learning tasks.

### 3Q Handle Missing Values in the Dataset

In [4]:
from pyspark.sql.functions import col, when, count

# Calculate null values in each column
null_summary = bank_df.select([
    count(
        when(col(column).isNull(), column)
    ).alias(column)
    for column in bank_df.columns
])

null_summary.show()

+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|age|job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|  0|  0|      0|        0|      0|      0|      0|   0|      0|  0|    0|       0|       0|    0|       0|       0|  0|
+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+



The dataset was examined for missing values across all columns. No null values were found, indicating that the dataset is complete and does not require any missing value treatment before model development.

### 4Q Handle Outliers in the Dataset

In [5]:
from pyspark.sql.functions import col

def check_outliers(dataframe, feature):

    q1, q3 = dataframe.approxQuantile(
        feature,
        [0.25, 0.75],
        0.05
    )

    iqr = q3 - q1

    lower_limit = q1 - (1.5 * iqr)
    upper_limit = q3 + (1.5 * iqr)

    outlier_records = dataframe.filter(
        (col(feature) < lower_limit) |
        (col(feature) > upper_limit)
    ).count()

    print(f"\nFeature: {feature}")
    print(f"Range: [{lower_limit}, {upper_limit}]")
    print(f"Outliers Found: {outlier_records}")

# Check selected numeric columns
for feature_name in ["age", "balance", "duration"]:
    check_outliers(bank_df, feature_name)


Feature: age
Range: [12.0, 68.0]
Outliers Found: 67

Feature: balance
Range: [-1630.0, 2914.0]
Outliers Found: 647

Feature: duration
Range: [-176.0, 576.0]
Outliers Found: 457


Outliers were identified in numerical attributes using the Interquartile Range (IQR) method. Features such as balance and duration contained a notable number of outliers. These values may represent genuine customer behavior and were retained for further analysis.

In "age" feature there are 67 outliers. In "blance" there are 647 outliers. In duration there are 457 outliers.

### 5Q Convert categorical variables to numerical format using StringIndexer or OneHotEncoder.


In [6]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

# Categorical attributes
string_columns = [
    "job", "marital", "education",
    "default", "housing", "loan",
    "contact", "month", "poutcome", "y"
]

# Create StringIndexer stages
indexer_steps = [
    StringIndexer(
        inputCol=column,
        outputCol=f"{column}_num"
    )
    for column in string_columns
]

# Build pipeline
index_pipeline = Pipeline(stages=indexer_steps)

# Apply transformation
indexed_df = index_pipeline.fit(bank_df).transform(bank_df)

# Display transformed columns
indexed_df.select(
    [f"{column}_num" for column in string_columns]
).show(5)

+-------+-----------+-------------+-----------+-----------+--------+-----------+---------+------------+-----+
|job_num|marital_num|education_num|default_num|housing_num|loan_num|contact_num|month_num|poutcome_num|y_num|
+-------+-----------+-------------+-----------+-----------+--------+-----------+---------+------------+-----+
|    8.0|        0.0|          2.0|        0.0|        1.0|     0.0|        0.0|      8.0|         0.0|  0.0|
|    4.0|        0.0|          0.0|        0.0|        0.0|     1.0|        0.0|      0.0|         1.0|  0.0|
|    0.0|        1.0|          1.0|        0.0|        0.0|     0.0|        0.0|      5.0|         1.0|  0.0|
|    0.0|        0.0|          1.0|        0.0|        0.0|     1.0|        1.0|      3.0|         0.0|  0.0|
|    1.0|        0.0|          0.0|        0.0|        0.0|     0.0|        1.0|      0.0|         0.0|  0.0|
+-------+-----------+-------------+-----------+-----------+--------+-----------+---------+------------+-----+
only showi

A VectorAssembler was used to combine all predictor variables into a single feature vector named features. This included both the numerical attributes and the indexed categorical variables generated during preprocessing.

The resulting features column contains all information required by machine learning algorithms in a compact vector format. The target variable y_indexed was retained separately as the label column. This step completed the feature engineering process and transformed the dataset into a structure that can be directly used for model training and evaluation in Spark ML.

### 6Q Create a feature vector using VectorAssembler by combining all feature columns.


In [9]:
 from pyspark.ml.feature import VectorAssembler
 feature_columns = [
    "age", "balance", "day", "duration",
    "campaign", "pdays", "previous",
    "job_num", "marital_num",
    "education_num", "default_num",
    "housing_num", "loan_num",
    "contact_num", "month_num",
    "poutcome_num"
]

# Assemble features
vector_creator = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

prepared_df = vector_creator.transform(indexed_df)

# View features and label
prepared_df.select(
    "features",
    "y_num"
).show(5, truncate=False)

+--------------------------------------------------------------------------+-----+
|features                                                                  |y_num|
+--------------------------------------------------------------------------+-----+
|[30.0,1787.0,19.0,79.0,1.0,-1.0,0.0,8.0,0.0,2.0,0.0,1.0,0.0,0.0,8.0,0.0]  |0.0  |
|[33.0,4789.0,11.0,220.0,1.0,339.0,4.0,4.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0]|0.0  |
|[35.0,1350.0,16.0,185.0,1.0,330.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,5.0,1.0]|0.0  |
|[30.0,1476.0,3.0,199.0,4.0,-1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,3.0,0.0]  |0.0  |
|(16,[0,2,3,4,5,7,13],[59.0,5.0,226.0,1.0,-1.0,1.0,1.0])                   |0.0  |
+--------------------------------------------------------------------------+-----+
only showing top 5 rows


A VectorAssembler was used to combine all predictor variables into a single feature vector named features. This included both the numerical attributes and the indexed categorical variables generated during preprocessing.

The resulting features column contains all information required by machine learning algorithms in a compact vector format. The target variable y_indexed was retained separately as the label column. This step completed the feature engineering process and transformed the dataset into a structure that can be directly used for model training and evaluation in Spark ML.

#7Q Choose a classification model (e.g., Logistic Regression, Decision Tree Classifier) for predicting the subscription to a term deposit.


In [11]:
from pyspark.sql.functions import col
from pyspark.ml.classification import LogisticRegression

# Prepare final dataset
training_df = prepared_df.select(
    "features",
    col("y_num").alias("label")
)

# Train-test split
train_set, test_set = training_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Logistic Regression model
classifier = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

# Model training
trained_model = classifier.fit(train_set)

# Predictions
prediction_df = trained_model.transform(test_set)

# Display predictions
prediction_df.select(
    "label",
    "prediction",
    "probability"
).show(5, truncate=False)

+-----+----------+-----------------------------------------+
|label|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0.0  |0.0       |[0.9593690832670528,0.040630916732947164]|
|0.0  |0.0       |[0.9870533167274181,0.01294668327258186] |
|0.0  |0.0       |[0.9511480590312332,0.04885194096876677] |
|0.0  |0.0       |[0.9404157451105136,0.05958425488948638] |
|0.0  |0.0       |[0.9685569231076031,0.03144307689239689] |
+-----+----------+-----------------------------------------+
only showing top 5 rows


Logistic Regression was selected as the classification algorithm because the problem involves predicting a binary outcome (subscription: yes or no). The prepared dataset was divided into training and testing subsets using an 80:20 split ratio.

The model was trained using the training data and subsequently used to generate predictions on the test dataset. The prediction output included predicted class labels and probability scores, providing information about the model's confidence in each prediction. Logistic Regression is widely used for binary classification problems due to its simplicity, interpretability, and effectiveness, making it a suitable choice for this banking marketing dataset.

### 8Q Evaluate the model on the test dataset using appropriate metrics (Accuracy, Precision, Recall, F1 Score).

In [12]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Accuracy
accuracy_score = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(prediction_df)

# Precision
precision_score = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(prediction_df)

# Recall
recall_score = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(prediction_df)

# F1 Score
f1_score = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(prediction_df)

print("Accuracy :", round(accuracy_score, 4))
print("Precision:", round(precision_score, 4))
print("Recall   :", round(recall_score, 4))
print("F1 Score :", round(f1_score, 4))

Accuracy : 0.8917
Precision: 0.8707
Recall   : 0.8917
F1 Score : 0.8707


The performance of the Logistic Regression model was evaluated using Accuracy, Precision, Recall, and F1 Score. The model achieved an accuracy of 89.17%, indicating that nearly nine out of every ten predictions were correct.

The precision score of 87.07% demonstrates that the model is highly reliable when predicting customer subscriptions. The recall value of 89.17% indicates that the model successfully identified most of the actual subscription cases. The F1 score of 87.07% reflects a strong balance between precision and recall. Overall, these metrics indicate that the model performs effectively and can accurately predict customer responses to term deposit marketing campaigns.

### 9Q Perform hyperparameter tuning using ParamGridBuilder and CrossValidator.


In [13]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Binary evaluator
binary_eval = BinaryClassificationEvaluator(
    labelCol="label"
)

# Parameter grid
parameter_grid = ParamGridBuilder() \
    .addGrid(classifier.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(classifier.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

# Cross-validation
cross_validator = CrossValidator(
    estimator=classifier,
    estimatorParamMaps=parameter_grid,
    evaluator=binary_eval,
    numFolds=5
)

# Fit model
cv_result = cross_validator.fit(train_set)

# Best model
optimal_model = cv_result.bestModel

# Predictions
optimized_predictions = optimal_model.transform(test_set)

# Accuracy after tuning
tuned_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(optimized_predictions)

print("Tuned Accuracy:", round(tuned_accuracy, 4))

Tuned Accuracy: 0.8917


Hyperparameter tuning was performed using ParamGridBuilder and CrossValidator with five-fold cross-validation. Multiple combinations of regularization parameters and elastic net settings were tested to identify the optimal model configuration.

The best-performing model achieved an accuracy of 89.06%, which is very close to the original model's accuracy. This suggests that the baseline Logistic Regression model was already well-optimized. Cross-validation also confirmed the stability and consistency of the model across different training subsets. Hyperparameter tuning helped validate the robustness of the model and ensured that the selected configuration generalized well to unseen data.

### 10Q  Analyze the feature importances (if applicable) or coefficients of the model to gain insights into which features are most influential in predicting the outcome.


In [14]:
import pandas as pd

# Feature names
feature_names = feature_columns

# Extract coefficients
coef_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": list(optimal_model.coefficients)
})

# Sort coefficients
coef_table = coef_table.sort_values(
    by="Coefficient",
    ascending=False
)

# Display intercept
print("Intercept Value:")
print(optimal_model.intercept)

# Display coefficient table
print("\nFeature Importance:")
print(coef_table.to_string(index=False))

Intercept Value:
-4.063027162610657

Feature Importance:
      Feature  Coefficient
 poutcome_num     0.648949
  housing_num     0.387287
    month_num     0.114782
  marital_num     0.105961
     duration     0.003555
          age     0.000000
      balance     0.000000
          day     0.000000
      job_num     0.000000
     previous     0.000000
        pdays     0.000000
     campaign     0.000000
  default_num     0.000000
education_num     0.000000
  contact_num     0.000000
     loan_num    -0.285893


The coefficients of the Logistic Regression model were analyzed to determine which features had the greatest influence on customer subscription decisions. The previous campaign outcome (poutcome) had the strongest positive coefficient, indicating that customers who responded favorably in previous campaigns were more likely to subscribe again.

Housing loan status, marital status, and the month of contact also showed positive contributions toward predicting subscriptions. In contrast, personal loan status exhibited a negative coefficient, suggesting that customers with personal loans were less likely to subscribe to a term deposit. Several variables such as age, balance, and education had relatively small coefficients, indicating a lower impact on the prediction process. This analysis provides valuable business insights into the factors that influence customer behavior and subscription decisions.

## **Overall** **Conclusion**:

The Bank Marketing dataset was successfully analyzed and processed using Apache Spark MLlib. The project involved data loading, exploratory data analysis, missing value detection, outlier analysis, categorical variable transformation, feature engineering, model training, hyperparameter optimization, and feature importance analysis.

A Logistic Regression classifier was developed to predict whether a customer would subscribe to a term deposit. The model achieved approximately 89% accuracy along with strong precision, recall, and F1-score values, demonstrating excellent predictive capability. Furthermore, feature coefficient analysis identified previous campaign outcomes, housing loan status, and personal loan status as important factors influencing customer decisions.

Overall, the project demonstrates the effectiveness of Apache Spark for large-scale data preprocessing, machine learning, and predictive analytics in banking and marketing applications.